In [1]:
import torch
from model.model import EncoderDecoderDAG
from utils.data import TranslateDataset, collate_fn, process_data
from torch.utils.data import DataLoader
from utils.load_tokenizer import load_tokenizer
from torch.optim import Adam
from typing import Tuple
from utils.fix_probs import fix_probs, masking
from tqdm.notebook import tqdm
from matplotlib import pyplot as plt
from utils.checkpoint import try_loading, epoch_resume, save_checkpoint
from utils.decoding import greedy_decoding, lookahead

g:\Projects\Visual Studio Code\LMTests\lmtest\lib\site-packages\transformers\utils\hub.py:123: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
tokenizer, vocab_size = load_tokenizer()

In [3]:
pad_idx = tokenizer.pad_token_id
eos_idx = tokenizer.eos_token_id

In [4]:
factor = 4
emb_size = 256
num_heads = 8
max_seq_len = 100
max_vertices = max_seq_len * factor

In [5]:
layers = [(2,2), 1, (1,1)]
out_layers = 1

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
device

device(type='cuda')

In [8]:
def create_model_fallback_fn() -> Tuple[EncoderDecoderDAG, Adam]:
    model = EncoderDecoderDAG(vocab_size, emb_size, num_heads, max_seq_len, max_vertices, layers, out_layers)
    model.to(device)
    optm = Adam(model.parameters(), lr=1e-3)
    return model, optm
    

In [9]:
checkpoint_dir = "./checkpoints"
checkpoint_name = "naivedag.pt"

In [10]:
model_class = EncoderDecoderDAG
optm_class = Adam

In [11]:
model, optm, losses, log_dir, tokens_seen = try_loading(checkpoint_dir, checkpoint_name, model_class, optm_class, device, create_model_fallback_fn)

Resuming, have seen 120,000 epochs and 44,268,635 tokens
Have 26704968 trainable parameters
Logging to runs/run_at_2023-12-24_13-34-45


In [12]:
en_test = "How are you"

In [13]:
encoded = tokenizer(en_test, return_tensors="pt").input_ids.to(device)
batch_size, l = encoded.shape
decoder_tokens = torch.arange(0, l * factor).unsqueeze(0).expand(batch_size, -1).to(device)
target_lens, vertex_lens, token_mask, vertex_mask = process_data(encoded, pad_idx, factor)
log_transition_probs, log_emission_probs = model(encoded, decoder_tokens, token_mask, vertex_mask)
mask = masking(log_transition_probs, vertex_lens)

In [14]:
log_transition_probs[0][2]

tensor([-28.0843, -25.6841, -29.7103, -26.7366, -27.3565, -29.1320, -30.3005,
        -30.4482, -30.2028, -26.3871, -28.6714, -18.9140, -19.2240, -29.4727,
        -24.2996,   0.0000], device='cuda:0', grad_fn=<SelectBackward0>)

In [23]:
#flog_transition_probs = log_transition_probs.masked_fill(mask !=0, float('-inf'))
flog_transition_probs = fix_probs(log_transition_probs, mask)
b1_transitions = flog_transition_probs[0]
b1_emissions = log_emission_probs[0]

In [24]:
b1_transitions[2]

tensor([    -inf,     -inf,     -inf, -26.7366, -27.3565, -29.1320, -30.3005,
        -30.4482, -30.2028, -26.3871, -28.6714, -18.9140, -19.2240, -29.4727,
        -24.2996,   0.0000], device='cuda:0', grad_fn=<SelectBackward0>)

In [25]:
torch.argmax(b1_transitions, dim=1)

tensor([15, 15, 15, 15, 15, 15, 15, 15,  9, 15, 15, 15, 15, 15, 15,  0],
       device='cuda:0')

In [26]:
torch.argmax(b1_emissions, dim=1)

tensor([65001, 32098,  6320,    44,   146, 32098,    69,    44, 32098,   904,
           23, 42791,   904,  6320, 32098,     0], device='cuda:0')

In [27]:
decoded = greedy_decoding(b1_transitions, b1_emissions, eos_idx)

In [28]:
tokenizer.decode(decoded)

'<s> </s>'

In [29]:
decoded2 = lookahead(b1_transitions, b1_emissions, eos_idx)

In [30]:
tokenizer.decode(decoded2)

'<s> </s>'